# IRIDE Burnt-Area Segmentation — NAS Training

This notebook orchestrates the full Neural Architecture Search (NAS) pipeline
for on-board burnt-area detection on the IRIDE constellation.

## Overview

The pipeline has **three stages**:

| Stage | What happens | Scope |
|-------|-------------|-------|
| **NAS search** | 50 architectures × 3 generations = 150 candidate models trained on an 8 k-tile stratified subset | Fast per-candidate evaluation |
| **FP32 fine-tune** | Best architecture re-trained from scratch on the full 50 k-tile dataset | Full accuracy |
| **FP16-aware fine-tune** | FP32 weights loaded, FP16 simulation applied for 15 more epochs | Myriad X deployment |

## Two saved models per candidate (and for the final winner)

- `model_fp32.pt` — standard FP32 weights → **ground-station / GPU deployment**
- `model_fp16aware.pt` — weights rounded to IEEE FP16 precision via forward hooks
  (STE gradient) → **Myriad X on-board deployment** (no hardware needed to produce this)

## Hardware target

Myriad X executes inference in IEEE 754 half precision.  Our FP16-aware training
simulates exactly this by rounding weights and activations to FP16 after each
Conv2d / Linear layer, while keeping gradients in FP32 (straight-through
estimator).  No OpenVINO export or physical device is required to produce a
Myriad-X-compatible model — the `.pt` file contains FP16-rounded weights that
are directly deployable.

## Dataset

88 112 tiles (82 GB) across:
- `train` / `val` / `test`       — PhiSat-2 simulated, scene-level split 80/10/10
- `heo_train` / `heo_val` / `heo_test` — real HEO acquisitions, held for fine-tuning

Class legend (7 classes including nodata):
```
0  clear          →  train ID 0
2  fresh burn     →  train ID 1   (≤90 days since fire — S4-08 target class)
3  old burn       →  train ID 2
4  cloud          →  train ID 3
5  cloud shadow   →  train ID 4
6  water          →  train ID 5
255 nodata        →  -1  (ignored by loss)
```

---
## Cell 1 — Notebook setup

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

In [2]:
%load_ext autoreload
%autoreload 2
    
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # must be before ANY torch import
print("CUDA_LAUNCH_BLOCKING set to 1")

CUDA_LAUNCH_BLOCKING set to 1


## Cell 3 — Environment check

Confirm GPU availability and PyNAS version before doing anything else.
You should see 2× NVIDIA A30-24C GPUs.

In [3]:
import torch
import pynas

print("PyNAS version :", pynas.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version  :", torch.version.cuda)
print("GPU count     :", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}:", torch.cuda.get_device_name(i))

PyNAS version : 0.1.0
CUDA available: True
CUDA version  : 13.0
GPU count     : 2
  GPU 0: NVIDIA A30-24C
  GPU 1: NVIDIA A30-24C


## Cell 4 — Imports

All project-local modules (`preprocessing`, `dataset`) must exist in the
project root before this cell runs.  They are written by the training setup
and should already be on disk at:
```
~/projects/iride_onboard-burnscar-mapper/preprocessing.py
~/projects/iride_onboard-burnscar-mapper/dataset.py
```

In [4]:
# ── Cell 3 — Imports ─────────────────────────────────────────────────────────
#
# preprocessing.py and dataset.py live in training/ (same folder as this notebook).
# pynas lives in pyqnas/src/ (vendored at project root).
# All data paths (tiles, models, EFFIS) are relative to PROJECT_ROOT.
# os.chdir(PROJECT_ROOT) ensures relative paths resolve correctly regardless
# of where JupyterLab was launched from.
# ─────────────────────────────────────────────────────────────────────────────

import sys, os, copy, json
import numpy as np
import pandas as pd
from pathlib import Path

# ── path setup ───────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.expanduser("~/projects/iride_onboard-burnscar-mapper")
TRAINING_DIR = os.path.join(PROJECT_ROOT, "training")

os.chdir(PROJECT_ROOT)                           # all relative paths from here
sys.path.insert(0, TRAINING_DIR)                 # preprocessing.py, dataset.py
sys.path.insert(0, PROJECT_ROOT)                 # effis_loader.py
sys.path.insert(0, os.path.join(PROJECT_ROOT, "pyqnas", "src"))   # pynas

# ── framework imports ─────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import WeightedRandomSampler

from pynas.core.config import load_default_config, default_config_path
from pynas.core.population import Population
from pynas.core.generic_lightning_module import GenericLightningSegmentationNetwork
from pynas.core.qat_utils import read_fp16aware_opts, prepare_fp16_aware

# ── local project modules ─────────────────────────────────────────────────────
from preprocessing import (
    to_reflectance,
    remap_mask_to_train_ids,
    train_augment,
    eval_transform,
    N_CLASSES,
    QUANTIFICATION,
)
from dataset import TileDataset

print("imports OK")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"TRAINING_DIR : {TRAINING_DIR}")
print(f"working dir  : {os.getcwd()}")
print(f"config path  : {default_config_path()}")

# quick verify the critical paths exist
for p in [TRAINING_DIR, f"{PROJECT_ROOT}/processed/dataset_v1",
          f"{PROJECT_ROOT}/pyqnas/src/pynas"]:
    status = "✓" if os.path.exists(p) else "✗ MISSING"
    print(f"  {status}  {p}")

imports OK
PROJECT_ROOT : /root/projects/iride_onboard-burnscar-mapper
TRAINING_DIR : /root/projects/iride_onboard-burnscar-mapper/training
working dir  : /root/projects/iride_onboard-burnscar-mapper
config path  : /root/projects/iride_onboard-burnscar-mapper/pyqnas/src/pynas/core/config.ini
  ✓  /root/projects/iride_onboard-burnscar-mapper/training
  ✓  /root/projects/iride_onboard-burnscar-mapper/processed/dataset_v1
  ✓  /root/projects/iride_onboard-burnscar-mapper/pyqnas/src/pynas


## Cell 5 — Load and verify the rewritten config

The config at `pyqnas/src/pynas/core/config.ini` was rewritten for this project:
- `[GA] population_size = 50` (50 models per generation)
- `[GA] max_iterations  = 3`  (3 generations → 150 total)
- `[Precision] train = fp32, fp16` (two training stages only, no INT8)
- `[Precision] fitness = fp16` (NAS selection driven by FP16-sim IoU)
- `[Myriad] enabled = false` (no hardware; GPU-only evaluation)

If any assertion fails, the config file was not written correctly.

In [5]:
config = load_default_config()

assert config.getint("GA", "population_size") == 50, \
    f"expected population_size=50, got {config.getint('GA','population_size')}"
assert config.getint("GA", "max_iterations")  == 2, \
    f"expected max_iterations=3, got {config.getint('GA','max_iterations')}"
assert config.getint("GA", "max_parameters")  == 3_000_000
assert config.get("Precision", "train").replace(" ", "") == "fp32,fp16"
assert config.get("Precision", "fitness") == "fp16"
assert not config.getboolean("Myriad", "enabled"), \
    "Myriad must be disabled — no hardware available"

print("Config verified ✓")
print(f"  population_size : {config.getint('GA', 'population_size')}")
print(f"  max_iterations  : {config.getint('GA', 'max_iterations')}")
print(f"  max_parameters  : {config.getint('GA', 'max_parameters'):,}  (= 6 MB at FP16)")
print(f"  fp32_epochs     : {config.getint('Precision', 'fp32_epochs')}")
print(f"  fp16_epochs     : {config.getint('QAT', 'finetune_epochs')}")
print(f"  train precisions: {config.get('Precision', 'train')}")
print(f"  fitness         : {config.get('Precision', 'fitness')}")
print(f"  myriad enabled  : {config.getboolean('Myriad', 'enabled')}")

Config verified ✓
  population_size : 30
  max_iterations  : 2
  max_parameters  : 2,500,000  (= 6 MB at FP16)
  fp32_epochs     : 12
  fp16_epochs     : 12
  train precisions: fp32, fp16
  fitness         : fp16
  myriad enabled  : False


## Cell 6 — Constants

All tunable numbers come from the config so the notebook never needs editing
to change run parameters — edit `config.ini` instead.

In [6]:
# ── Reproducibility ───────────────────────────────────────────────────────────
pl.seed_everything(seed=config.getint("Computation", "seed"), workers=True)
torch.set_float32_matmul_precision("medium")  # speeds up matmul on A30 with no IoU impact

# ── NAS / GA hyper-parameters ────────────────────────────────────────────────
MAX_LAYERS    = config.getint("NAS",    "max_layers",      fallback=7)
N_INDIVIDUALS = config.getint("GA",     "population_size")
K_BEST        = config.getint("GA",     "k_best")
N_RANDOM      = config.getint("GA",     "n_random")
POOL_CUTOFF   = config.getfloat("GA",   "mating_pool_cutoff")
MUTATION_P    = config.getfloat("GA",   "mutation_probability")
MAX_ITER      = config.getint("GA",     "max_iterations")
BATCH_SIZE    = config.getint("GA",     "batch_size")
MAX_PARAMS    = config.getint("GA",     "max_parameters")

# ── Training epochs ──────────────────────────────────────────────────────────
FP32_EPOCHS   = config.getint("Precision", "fp32_epochs")
FP16_EPOCHS   = config.getint("QAT",       "finetune_epochs")

# ── Optimiser ────────────────────────────────────────────────────────────────
LR            = 10 ** config.getfloat("Search Space", "default_log_lr", fallback=-3.0)

# ── Dataset ──────────────────────────────────────────────────────────────────
N_BANDS       = 7
TILE_SIZE     = 256
TILES_ROOT    = "processed/dataset_v1"
INDEX_CSV     = f"{TILES_ROOT}/tiles_index.csv"
SAVE_DIR      = "models_traced"
NAS_SUBSET_N  = 12_000   # tiles used for NAS search evaluation (full = ~50 k)

# ── Class weights for cross-entropy loss ─────────────────────────────────────
# Derived from dataset_stats_summary.csv (Cell 9 of the stats notebook).
# Inverse pixel frequency, normalised to sum to 1.
# clear=66%  fresh=15%  old=6.5%  cloud=0.4%  shadow=0.1%  water=11%
# Inverse-frequency class weights, capped at 10× clear's raw weight.
# Without cap: shadow=0.78, cloud=0.19 → 97% of gradient on two rare classes
# → model collapses to predicting one class everywhere.
CLASS_PIXEL_FREQS = np.array([0.661, 0.154, 0.065, 0.004, 0.001, 0.110])
_raw    = 1.0 / np.maximum(CLASS_PIXEL_FREQS, 1e-4)
_cap    = _raw[0] * 10.0          # clear's inverse freq × 10
_capped = np.minimum(_raw, _cap)
CLASS_WEIGHTS = torch.tensor([
    0.05,   # clear
    0.25,   # fresh_burn  ← primary target
    0.35,   # old_burn    ← hardest class
    0.12,   # cloud
    0.12,   # shadow
    0.11,   # water
], dtype=torch.float32)

print("Class weights:")
for name, w in zip(["clear","fresh_burn","old_burn","cloud","shadow","water"], CLASS_WEIGHTS):
    print(f"  {name:<14}: {w:.4f}")
print(f"  sum: {CLASS_WEIGHTS.sum():.4f}")

Seed set to 42


Class weights:
  clear         : 0.0500
  fresh_burn    : 0.2500
  old_burn      : 0.3500
  cloud         : 0.1200
  shadow        : 0.1200
  water         : 0.1100
  sum: 1.0000


## Cell 7 — DataModule

`BalancedTileDataModule` replaces the zarr-based `SegmentationDataModule`
from the original py-q-nas notebook.

Key design choices:
- **Weighted tile sampler at train time**: each tile's sampling probability
  is proportional to its rarest-class pixel fraction (fresh/old burn, cloud,
  shadow, water).  Pure-clear tiles get a floor weight of 0.02 so they're
  never completely absent.  This means the model sees burn/cloud examples
  far more often than their 6–15% pixel share would normally allow.
- **NAS subset mode** (`nas_subset_n`): for the search phase, a stratified
  8 k-tile subset is used so each candidate trains in minutes rather than
  hours.  All informative tiles are included; clear tiles fill the remainder.
- **Full dataset mode** (`nas_subset_n=None`): used for final winner
  retraining after the search completes.

The datamodule exposes `input_shape` and `num_classes` which are read by
`Population.build_model` to construct the right network head.

In [7]:
class BalancedTileDataModule(pl.LightningDataModule):
    def __init__(self, tiles_root, index_csv, batch_size=16, num_workers=4,
                 nas_subset_n=None, seed=42):
        super().__init__()
        self.root        = tiles_root
        self.index_csv   = index_csv
        self.batch_size  = batch_size
        self.num_workers = num_workers
        self.nas_n       = nas_subset_n
        self.seed        = seed
        self.input_shape = (N_BANDS, TILE_SIZE, TILE_SIZE)
        self.num_classes = N_CLASSES

    def setup(self, stage=None):
        idx   = pd.read_csv(self.index_csv)
        train = idx[(idx.split == "train") & (idx.gsd == "native")].reset_index(drop=True)
        val   = idx[(idx.split == "val")   & (idx.gsd == "native")].reset_index(drop=True)
        test  = idx[(idx.split == "test")  & (idx.gsd == "native")].reset_index(drop=True)

        if self.nas_n is not None:
            rng     = np.random.default_rng(self.seed)
            inf_idx = train[train.informative].index.tolist()
            clr_idx = train[~train.informative].index.tolist()
            # 80% informative tiles, 20% clear — both capped at nas_n budget
            n_inf = min(len(inf_idx), int(self.nas_n * 0.8))
            n_clr = min(len(clr_idx), self.nas_n - n_inf)
            chosen_inf = rng.choice(inf_idx, size=n_inf, replace=False).tolist()
            chosen_clr = rng.choice(clr_idx, size=n_clr, replace=False).tolist()
            train = train.loc[chosen_inf + chosen_clr].reset_index(drop=True)
            print(f"NAS subset: {len(train):,} tiles "
                 f"({n_inf:,} informative + {n_clr:,} clear)")

        self.train_ds = TileDataset(self.root, train, is_train=True)
        self.val_ds   = TileDataset(self.root, val,   is_train=False)
        self.test_ds  = TileDataset(self.root, test,  is_train=False)

        # Tile sampling weight = combined rare-class pixel fraction.
        # Cap at 0.3 so burn-heavy tiles don't monopolise every batch —
        # the model still needs to see clear land to avoid false positives.
        # Without the cap, combined with high cloud/shadow loss weights,
        # the gradient signal becomes chaotic and the model collapses.
        rare = (train.get("frac_2", 0) + train.get("frac_3", 0) +
                train.get("frac_4", 0) + train.get("frac_5", 0)).values
        w = np.clip(np.maximum(rare, 0.02), 0, 0.3).astype(np.float64)
        w /= w.sum()
        self._sampler = WeightedRandomSampler(
            torch.from_numpy(w).float(), len(self.train_ds), replacement=True)

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_ds, batch_size=self.batch_size, sampler=self._sampler,
            num_workers=self.num_workers, pin_memory=True, drop_last=True)

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_ds, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return torch.utils.data.DataLoader(
            self.test_ds, batch_size=self.batch_size, shuffle=False,
            num_workers=self.num_workers, pin_memory=True)

## Cell 8 — Sanity check

Verify the datamodule, tile shapes, reflectance range, and mask values
before spending any GPU time.  **Do not proceed to Cell 9 if this fails.**

In [8]:
dm_nas = BalancedTileDataModule(
    TILES_ROOT, INDEX_CSV,
    batch_size=BATCH_SIZE, num_workers=4,
    nas_subset_n=NAS_SUBSET_N,
)
dm_nas.setup()

img, mask = next(iter(dm_nas.train_dataloader()))

print(f"img  shape : {tuple(img.shape)}   dtype: {img.dtype}")
print(f"mask shape : {tuple(mask.shape)}  dtype: {mask.dtype}")
print(f"img  range : [{img.min():.4f}, {img.max():.4f}]  (expect ~[0, 1])")
print(f"mask values: {sorted(mask.unique().tolist())}  (expect subset of {{-1,0,1,2,3,4,5}})")
print(f"train tiles: {len(dm_nas.train_ds):,}   val: {len(dm_nas.val_ds):,}   test: {len(dm_nas.test_ds):,}")

assert tuple(img.shape[1:]) == (N_BANDS, TILE_SIZE, TILE_SIZE), \
    f"expected ({N_BANDS},{TILE_SIZE},{TILE_SIZE}), got {tuple(img.shape[1:])}"
assert img.max() <= 1.01,  "reflectance > 1.0 — normalisation bug in preprocessing.py"
assert img.min() >= -0.01, "reflectance < 0.0 — normalisation bug in preprocessing.py"
assert set(mask.unique().tolist()) <= set(range(-1, N_CLASSES)), \
    f"unexpected mask values: {mask.unique().tolist()}"

print("\n✓ Sanity check passed — safe to proceed to NAS search")

NAS subset: 12,000 tiles (9,600 informative + 2,400 clear)
img  shape : (8, 7, 256, 256)   dtype: torch.float32
mask shape : (8, 256, 256)  dtype: torch.int64
img  range : [0.0000, 1.0000]  (expect ~[0, 1])
mask values: [-1, 0, 1, 2, 3, 4, 5]  (expect subset of {-1,0,1,2,3,4,5})
train tiles: 12,000   val: 6,367   test: 5,865

✓ Sanity check passed — safe to proceed to NAS search


## Cell 9 — Two-scenario training function

This function replaces `pop.train_generation()` from the original notebook.
The original ran 6 scenarios (3 GPU + 3 Myriad) per candidate.
We run exactly **2** (both GPU-only):

1. **FP32** — trains from random init for `fp32_epochs`.
   Saves `model_fp32.pt` → ground/GPU deployment.

2. **FP16-aware finetune** — loads FP32 weights, applies `prepare_fp16_aware`
   (registers STE hooks on Conv2d/Linear to round activations/weights to
   IEEE FP16 each forward pass), trains for `fp16_epochs` at 0.1× LR.
   Saves `model_fp16aware.pt` → Myriad X deployment.

Fitness = α · fp16_iou + β · min(fps / fps_target, 1)
with α=1.0, β=0.2, fps_target=30  (same as [Fitness] section of config).

In [9]:
# ── Cell 8 — Two-scenario training function ──────────────────────────────────
#
# Replaces pop.train_generation() from the original py-q-nas notebook.
# Runs exactly TWO GPU-only scenarios per candidate:
#
#   Scenario 1 — FP32 from scratch
#     Full training for fp32_epochs. Saves model_fp32.pt.
#     Target: ground station / GPU deployment.
#
#   Scenario 2 — FP16-aware fine-tune from FP32
#     Loads FP32 weights, attaches STE-based FP16 rounding hooks to every
#     Conv2d and Linear layer (IEEE 754 half precision, forward-only),
#     trains for fp16_epochs at 0.1× LR. Saves model_fp16aware.pt.
#     Target: Myriad X on-board deployment (no hardware required to produce).
#
#   Fitness = α·fp16_iou + β·min(fps / fps_target, 1)
#   with α=1.0  β=0.2  fps_target=30  (matches [Fitness] section of config)
#
# Robustness features:
#   - CUDA context is synchronised and cache-cleared after every failure so a
#     single OOM or assert cannot poison subsequent candidates in the same run.
#   - All results written to metrics.json + pop.df immediately, so a kernel
#     restart never loses a completed candidate.
# ─────────────────────────────────────────────────────────────────────────────

def free_vram_gb(device=0):
    """Returns free VRAM in GB on the given CUDA device."""
    if not torch.cuda.is_available():
        return 0.0
    torch.cuda.synchronize(device)
    free, _ = torch.cuda.mem_get_info(device)
    return free / 1e9


MIN_FREE_VRAM_GB = 4.0   # skip candidate if less than this is available


def _reset_cuda():
    """Best-effort CUDA context flush after a failure."""
    try:
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
    except Exception:
        pass
    try:
        import gc; gc.collect()
    except Exception:
        pass


def train_two_scenarios(
    pop,
    idx,
    dm,
    lr=None,
    fp32_epochs=None,
    fp16_epochs=None,
):
    """
    Train one NAS candidate through FP32 + FP16-aware scenarios.

    Parameters
    ----------
    pop         : Population  — the active NAS population object
    idx         : int         — index into pop.population
    dm          : LightningDataModule — provides train/val/test dataloaders
    lr          : float | None — base learning rate; defaults to LR from Cell 5
    fp32_epochs : int | None  — overrides FP32_EPOCHS (use for final retraining)
    fp16_epochs : int | None  — overrides FP16_EPOCHS (use for final retraining)

    Returns
    -------
    dict with keys fp32_iou, fp32_fps, fp16_iou, fp16_fps, fitness, or None on failure.
    """
    # resolve defaults from outer scope (Cell 5 constants)
    _lr          = lr          if lr          is not None else LR
    _fp32_epochs = fp32_epochs if fp32_epochs is not None else FP32_EPOCHS
    _fp16_epochs = fp16_epochs if fp16_epochs is not None else FP16_EPOCHS

    individual = pop.population[idx]
    gen        = pop.generation
    gen_dir    = Path(SAVE_DIR) / f"generation_{gen}" / f"model_{idx}"
    gen_dir.mkdir(parents=True, exist_ok=True)

    # class-weighted loss; ignore_index=-1 keeps nodata pixels out of the gradient
    loss_fn = nn.CrossEntropyLoss(
        weight     = CLASS_WEIGHTS.to(torch.float32).cuda(),
        ignore_index = -1,
        reduction  = "mean",
    )

    # ── VRAM pre-check ────────────────────────────────────────────────────
    free = free_vram_gb(0)
    if free < MIN_FREE_VRAM_GB:
        _reset_cuda()
        free = free_vram_gb(0)
    if free < MIN_FREE_VRAM_GB:
        print(f"  [{idx:02d}] only {free:.1f} GB free — skipping "
              f"(will be retried next resume)")
        return None

    # ═════════════════════════════════════════════════════════════════════
    # Scenario 1 — FP32
    # ═════════════════════════════════════════════════════════════════════
    print(f"  [{idx:02d}] Scenario 1: FP32  ({_fp32_epochs} epochs, lr={_lr:.2e})")
    try:
        model, valid = pop.build_model(
            individual.parsed_layers, task="segmentation")
        if not valid:
            print(f"  [{idx:02d}] build_model returned invalid — skipping")
            individual.fitness = 0.0
            return None

        LM         = GenericLightningSegmentationNetwork(
            model=model, learning_rate=_lr)
        LM.loss_fn = loss_fn

        trainer = pl.Trainer(
            max_epochs              = _fp32_epochs,
            accelerator             = "gpu",
            devices                 = 1,
            accumulate_grad_batches = 2,      # effective batch = NAS_BATCH_SIZE×2
            logger                  = False,
            
            enable_checkpointing    = False,
            enable_progress_bar     = False,   # ← turn off per-step bar
            log_every_n_steps       = 999,     # ← suppress step-level logging
            callbacks               = [], 
        )
        trainer.fit(LM, dm)
        res_fp32  = trainer.test(LM, dm, verbose=False)[0]
        fp32_iou  = float(res_fp32.get("test_iou",
                          res_fp32.get("test_iou_epoch", 0.0)))
        fp32_fps  = float(res_fp32.get("test_fps", 1.0))

        # save FP32 weights and keep a CPU copy for FP16 init
        torch.save(LM.model.state_dict(), gen_dir / "model_fp32.pt")
        fp32_state = {k: v.cpu().clone() for k, v in LM.model.state_dict().items()}
        print(f"  [{idx:02d}] FP32  → IoU={fp32_iou:.4f}  FPS={fp32_fps:.1f}")

        # free GPU memory before building the FP16 model
        del LM, trainer, model
        _reset_cuda()

    except torch.cuda.OutOfMemoryError:
        print(f"  [{idx:02d}] FP32 OOM — skipping (increase accumulate_grad_batches)")
        individual.fitness = 0.0
        _reset_cuda()
        return None
    except Exception as e:
        print(f"  [{idx:02d}] FP32 FAILED: {e}")
        individual.fitness = 0.0
        _reset_cuda()
        return None

    # ═════════════════════════════════════════════════════════════════════
    # Scenario 2 — FP16-aware fine-tune
    # ═════════════════════════════════════════════════════════════════════
    print(f"  [{idx:02d}] Scenario 2: FP16-aware "
          f"({_fp16_epochs} epochs, lr={_lr*0.1:.2e})")
    try:
        fp16_opts = read_fp16aware_opts(config)
        fp16_opts.finetune_epochs = _fp16_epochs

        # fresh model instance; load FP32 weights; attach FP16 simulation hooks
        model_ft, _ = pop.build_model(
            individual.parsed_layers, task="segmentation")
        model_ft.load_state_dict(fp32_state)
        ctx = prepare_fp16_aware(model_ft, fp16_opts)

        LM_ft         = GenericLightningSegmentationNetwork(
            model=model_ft, learning_rate=_lr * 0.1)
        LM_ft.loss_fn = loss_fn

        trainer_ft = pl.Trainer(
            max_epochs              = _fp16_epochs,
            accelerator             = "gpu",
            devices                 = 1,
            accumulate_grad_batches = 2,
            logger                  = False,
              enable_checkpointing    = False,
            enable_progress_bar     = False,   # ← turn off per-step bar
            log_every_n_steps       = 999,     # ← suppress step-level logging
            callbacks               = [], 
        )
        trainer_ft.fit(LM_ft, dm)
        res_fp16  = trainer_ft.test(LM_ft, dm, verbose=False)[0]
        fp16_iou  = float(res_fp16.get("test_iou",
                          res_fp16.get("test_iou_epoch", 0.0)))
        fp16_fps  = float(res_fp16.get("test_fps", 1.0))

        torch.save(LM_ft.model.state_dict(), gen_dir / "model_fp16aware.pt")
        ctx.close()   # detach STE hooks and weight parametrizations cleanly
        print(f"  [{idx:02d}] FP16  → IoU={fp16_iou:.4f}  FPS={fp16_fps:.1f}")

        del LM_ft, trainer_ft, model_ft
        _reset_cuda()

    except torch.cuda.OutOfMemoryError:
        print(f"  [{idx:02d}] FP16 OOM — saving FP32 only, fitness=fp32_iou")
        fp16_iou, fp16_fps = fp32_iou * 0.95, fp32_fps  # penalised estimate
        _reset_cuda()
    except Exception as e:
        print(f"  [{idx:02d}] FP16 FAILED: {e} — saving FP32 only")
        fp16_iou, fp16_fps = fp32_iou * 0.95, fp32_fps
        _reset_cuda()

    # ═════════════════════════════════════════════════════════════════════
    # Fitness + bookkeeping
    # ═════════════════════════════════════════════════════════════════════
    alpha, beta, fps_target = 1.0, 0.2, 30.0
    fitness = alpha * fp16_iou + beta * min(fp16_fps / fps_target, 1.0)

    individual.fitness = fitness
    individual.metric  = fp16_iou
    individual.fps     = fp16_fps

    pop.df.loc[idx, "fp32_gpu_iou"] = fp32_iou
    pop.df.loc[idx, "fp32_gpu_fps"] = fp32_fps
    pop.df.loc[idx, "fp16_gpu_iou"] = fp16_iou
    pop.df.loc[idx, "fp16_gpu_fps"] = fp16_fps
    pop.df.loc[idx, "Fitness"]      = fitness
    pop.df.loc[idx, "GPU_IoU"]      = fp16_iou
    pop.df.loc[idx, "GPU_FPS"]      = fp16_fps

    metrics = {
        "gen":      gen,
        "idx":      idx,
        "params":   individual.model_size,
        "fp32_iou": fp32_iou,
        "fp32_fps": fp32_fps,
        "fp16_iou": fp16_iou,
        "fp16_fps": fp16_fps,
        "fitness":  fitness,
        "fp32_model": str(gen_dir / "model_fp32.pt"),
        "fp16_model": str(gen_dir / "model_fp16aware.pt"),
    }
    json.dump(metrics, open(gen_dir / "metrics.json", "w"), indent=2)

    # checkpoint after every candidate so restarts lose at most one model
    pop.save_dataframe()
    pop.save_population()

    print(f"  [{idx:02d}] DONE  — params={individual.model_size:,}  "
          f"fp32={fp32_iou:.4f}  fp16={fp16_iou:.4f}  "
          f"fitness={fitness:.4f}  VRAM_free={free_vram_gb(0):.1f}GB")
    return metrics


print("train_two_scenarios defined ✓")
print(f"  NAS search : {FP32_EPOCHS} FP32 + {FP16_EPOCHS} FP16 epochs, "
      f"batch {BATCH_SIZE}×2 accumulation")
print(f"  Final train: 50 FP32 + 15 FP16 epochs (Cell 11)")

train_two_scenarios defined ✓
  NAS search : 12 FP32 + 12 FP16 epochs, batch 8×2 accumulation
  Final train: 50 FP32 + 15 FP16 epochs (Cell 11)


## Cell 10 — Create population

`Population` generates `n_individuals` random candidate architectures and
validates that each fits within the `max_parameters` budget.
`initial_poll()` triggers this generation and saves a checkpoint.

**Expected output:** a progress bar while 50 architectures are generated.

In [ ]:
# ── Cell 10 — Create population
# Uses a lightweight stub datamodule during initial_poll so tile data
# is never loaded into RAM during architecture generation/validation.
# The real datamodule (dm_nas) is only created in Cell 11 right before training.

import pickle, gc

class LightDM:
    """Stub datamodule — gives Population what it needs to build and validate
    architectures (input shape + num_classes) without loading any tile data.
    Prevents the 30GB+ RAM spike that was OOM-killing the process during
    initial_poll when dm_nas was passed directly."""
    input_shape = (N_BANDS, TILE_SIZE, TILE_SIZE)
    num_classes = N_CLASSES

pop = Population(
    n_individuals  = N_INDIVIDUALS,
    max_layers     = MAX_LAYERS,
    dm             = LightDM(),        # lightweight stub — no tile data loaded
    max_parameters = MAX_PARAMS,
    save_directory = SAVE_DIR,
)
pop.cfg = config   # must be set before initial_poll so QAT opts are readable

# ── Resume from checkpoint if one exists (survives kernel restarts) ──────────
checkpoint_pkl = f"{SAVE_DIR}/src/population_0.pkl"
checkpoint_df  = f"{SAVE_DIR}/src/df_population_0.pkl"
if os.path.exists(checkpoint_pkl):
    pop.population = pickle.load(open(checkpoint_pkl, "rb"))
    pop.df         = pd.read_pickle(checkpoint_df)
    n_done = pop.df["Fitness"].notna().sum() if "Fitness" in pop.df.columns else 0
    print(f"Resumed from checkpoint: {len(pop.population)} individuals, "
          f"{n_done} already trained")
else:
    gc.collect()   # clear RAM before building architectures
    pop.initial_poll()
    print(f"Fresh population: {len(pop.population)} individuals")

print(pop.df[["Generation", "Params"]].describe())
assert len(pop.population) >= int(N_INDIVIDUALS * 0.8), \
    f"population too small: {len(pop.population)}/{N_INDIVIDUALS}"
print(f"Population ready: {len(pop.population)}/{N_INDIVIDUALS} ✓")

# ── Now create the real datamodule — AFTER population is loaded ───────────────
# This ensures tile data never competes with architecture RAM during initial_poll
dm_nas = BalancedTileDataModule(
    TILES_ROOT, INDEX_CSV,
    batch_size=BATCH_SIZE, num_workers=2,
    nas_subset_n=NAS_SUBSET_N,
)
dm_nas.setup()
print(f"NAS datamodule ready: {len(dm_nas.train_ds):,} train tiles  "
      f"val: {len(dm_nas.val_ds):,}  test: {len(dm_nas.test_ds):,}")

Generating Population:   0%|                                                                    | 0/30 [00:00<?, ?it/s]

## Cell 11 — NAS search loop

Runs `MAX_ITER=3` generations, each training all 50 candidates through the
two-scenario pipeline.

**Total:** 3 × 50 = 150 model trainings.
**Resume-safe:** if the kernel crashes, re-running this cell skips any
candidate that already has a valid `Fitness` value in `pop.df`.

At the end of each generation except the last, `pop.evolve()` applies
crossover + mutation to produce the next generation's 50 candidates,
carrying forward the `k_best=2` elite models unchanged.

In [ ]:
# ── Cell 11 — NAS search loop
# 3 generations × N_INDIVIDUALS candidates = total model trainings.
# Resume-safe: re-running this cell skips any candidate that already
# has a valid Fitness value in pop.df.
# STOP_ON_FIRST_ERROR=True stops at the first failure with a full
# synchronous traceback (requires CUDA_LAUNCH_BLOCKING=1 in Cell 1).
# Set to False once training is confirmed working.

STOP_ON_FIRST_ERROR = False   # set True only for debugging

for gen_idx in range(MAX_ITER):
    print(f"\n{'='*65}")
    print(f"  GENERATION {pop.generation + 1} / {MAX_ITER} "
          f"— {len(pop.population)} candidates")
    print(f"{'='*65}")

    first_error_idx = None

    for idx in range(len(pop.population)):
        # resume-safe: skip if already trained this generation
        if (pop.df is not None and
            "Fitness" in pop.df.columns and
            idx < len(pop.df) and
            not pd.isna(pop.df.loc[idx, "Fitness"])):
            print(f"  [{idx:02d}] already trained — skipping")
            continue

        # VRAM guard
        free = free_vram_gb(0)
        if free < MIN_FREE_VRAM_GB:
            torch.cuda.empty_cache()
            free = free_vram_gb(0)
        if free < MIN_FREE_VRAM_GB:
            print(f"  [{idx:02d}] only {free:.1f} GB VRAM free — skipping")
            continue

        result = train_two_scenarios(pop, idx, dm=dm_nas)

        if result is None and STOP_ON_FIRST_ERROR:
            first_error_idx = idx
            print(f"\n!! Stopped at idx={idx}")
            print(f"   Check traceback above for root cause.")
            print(f"   Fix issue, set STOP_ON_FIRST_ERROR=False, restart, rerun.")
            break

    # sort and checkpoint after every generation
    pop._sort_population()
    best = pop.population[0]
    best_fitness = best.fitness    if best.fitness    is not None else float("nan")
    best_iou     = best.metric     if best.metric     is not None else float("nan")
    best_params  = best.model_size if best.model_size is not None else 0
    print(f"\nGeneration {pop.generation} complete.")
    print(f"  Best: fitness={best_fitness:.4f}  fp16_iou={best_iou:.4f}  "
         f"params={best_params:,}")
    pop._checkpoint()

    if first_error_idx is not None:
        break   # don't evolve on a poisoned generation

    if gen_idx < MAX_ITER - 1:
        pop.evolve(
            mating_pool_cutoff  = POOL_CUTOFF,
            mutation_probability= MUTATION_P,
            k_best              = K_BEST,
            n_random            = N_RANDOM,
        )
        print(f"Evolution complete — new generation ready")

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print("NAS SEARCH COMPLETE" if not STOP_ON_FIRST_ERROR else "NAS SEARCH — DIAGNOSTIC RUN")
print("="*65)
if pop.df is not None and len(pop.df) > 0:
    avail = [c for c in ["Generation","Params","Fitness","fp32_gpu_iou","fp16_gpu_iou"]
             if c in pop.df.columns]
    print(pop.df[avail].sort_values("Fitness", ascending=False).head(10).to_string())
else:
    print("No completed candidates yet.")

## Cell 12 — Retrain winner on the full dataset

The NAS search used only 8 000 tiles per candidate for speed.
Here we take the winning architecture and retrain it properly:

- **FP32 full training**: 50 epochs on all 50 822 train tiles
- **FP16-aware finetune**: 15 epochs from the FP32 checkpoint

This is the step that produces the final deployable models.

In [ ]:
# full dataset datamodule (no NAS subset limit)
dm_full = BalancedTileDataModule(
    TILES_ROOT, INDEX_CSV,
    batch_size=BATCH_SIZE, num_workers=4,
    nas_subset_n=None,   # use ALL train tiles
)
dm_full.setup()
print(f"Full dataset — train: {len(dm_full.train_ds):,}  "
      f"val: {len(dm_full.val_ds):,}  test: {len(dm_full.test_ds):,}")

# inspect winner
winner = pop.population[0]
print(f"\nWinner architecture:")
print(f"  fitness  : {winner.fitness:.4f}")
print(f"  fp16 IoU : {winner.metric:.4f}")
print(f"  params   : {winner.model_size:,}  "
      f"({winner.model_size*2/1e6:.2f} MB at FP16)")
print(f"  layers   : {winner.parsed_layers}")

# retrain with higher epoch budgets
print("\nStarting full retraining...")
train_two_scenarios(
    pop, idx=0, dm=dm_full,
    lr=LR,
    fp32_epochs=50,   # full convergence budget
    fp16_epochs=15,   # FP16 fine-tune
)

final_dir = Path(SAVE_DIR) / f"generation_{pop.generation}" / "model_0"
print("\nFinal models saved:")
print(f"  FP32 (ground/GPU deployment) : {final_dir / 'model_fp32.pt'}")
print(f"  FP16 (Myriad X deployment)   : {final_dir / 'model_fp16aware.pt'}")

## Cell 13 — Verify saved models

Confirm both model files exist, load cleanly, and that the FP16 model
fits within the 6 MB Myriad X target.

In [ ]:
fp32_path = final_dir / "model_fp32.pt"
fp16_path = final_dir / "model_fp16aware.pt"

assert fp32_path.exists(), f"FP32 model not found: {fp32_path}"
assert fp16_path.exists(), f"FP16 model not found: {fp16_path}"

fp32_sd = torch.load(fp32_path, map_location="cpu")
fp16_sd = torch.load(fp16_path, map_location="cpu")

n_params  = sum(v.numel() for v in fp32_sd.values())
fp32_mb   = n_params * 4 / 1e6    # FP32 = 4 bytes/param
fp16_mb   = n_params * 2 / 1e6    # FP16 = 2 bytes/param

print(f"Parameters   : {n_params:,}")
print(f"FP32 size    : {fp32_mb:.2f} MB  (ground/GPU deployment)")
print(f"FP16 size    : {fp16_mb:.2f} MB  (Myriad X on-board deployment)")
print(f"6 MB target  : {'✓ PASS' if fp16_mb <= 6.0 else '✗ FAIL — architecture too large'}")

assert fp16_mb <= 6.5, \
    f"FP16 model exceeds 6 MB budget: {fp16_mb:.2f} MB — lower max_parameters in config.ini"

# load metrics JSON for a clean summary
metrics = json.load(open(final_dir / "metrics.json"))
print(f"\nFinal model performance (full dataset, NAS-search split):")
print(f"  FP32 IoU : {metrics['fp32_iou']:.4f}")
print(f"  FP16 IoU : {metrics['fp16_iou']:.4f}")
print(f"  FP16 loss: {metrics['fp32_iou'] - metrics['fp16_iou']:.4f}  "
      f"(quantisation gap — target <0.01)")

## Cell 14 — Full search results summary

Inspect every candidate across all generations.

In [ ]:
all_metrics = []
for metrics_path in sorted(Path(SAVE_DIR).glob("generation_*/model_*/metrics.json")):
    try:
        m = json.load(open(metrics_path))
        all_metrics.append(m)
    except Exception:
        pass

results_df = pd.DataFrame(all_metrics).sort_values("fitness", ascending=False)
print(f"Total candidates evaluated: {len(results_df)}")
print(f"\nTop 10 by fitness:")
print(results_df[["gen","idx","params","fp32_iou","fp16_iou","fitness"]].head(10).to_string(index=False))

# save full results
results_df.to_csv(f"{SAVE_DIR}/nas_results_all.csv", index=False)
print(f"\nFull results saved to {SAVE_DIR}/nas_results_all.csv")